# Projekt z przedmiotu *Eksploracja Danych*

## Etap 2:  Przygotowanie danych + Modelowanie

### Analizowany zbiór danych: **Brewer's Friend Beer Recipes**

#### Autorzy:
- Anna Sztukowska 188803
- Michał Sugalski 193290
- Lucjan Gackowski 193150


#### Ogólny opis zbioru

Zbiór **Brewer's Friend Beer Recipes** zawiera dane dotyczące domowych receptur piwa udostępnianych przez użytkowników platformy Brewer's Friend – narzędzia wspierającego amatorskich i półprofesjonalnych piwowarów. Każdy wiersz odpowiada jednej recepturze i zawiera ogólne parametry techniczne związane z procesem warzenia.
Dane obejmują szeroki zakres ogólnych parametrów warzenia, takich jak styl piwa, zawartość alkoholu (`ABV`), poziom goryczki (`IBU`), kolor (`SRM`), gęstość początkowa (`OG`) i końcowa (`FG`), metoda warzenia (np. all grain, extract), objętości na różnych etapach produkcji, a także temperatury fermentacji.
Dane mają postać numeryczną lub kategoryczną i mogą służyć do analizy trendów, porównań stylów piwa, klasteryzacji receptur lub budowy modeli predykcyjnych opartych na parametrach fizykochemicznych trunku.

#### Charakterystyka zbioru danych
- **Pochodzenie:** Dane zostały zebrane z platformy Brewer's Friend i udostępnione na Kaggle przez użytkownika jtrofe.
- **Format:** `.csv`
- **Liczba przykładów:** 73 861 receptur piwa
- **Liczba atrybutów:** 23 kolumny opisujące właściwości każdej receptury
- **Struktura:** Zbiór składa się z dwóch plików:
  - `recipeData.csv` – główny zbiór zawierający informacje o recepturach piwa
  - `styleData.csv` – uzupełniający zbiór zawierający opisy stylów piwa

#### Określenie celu eksploracji i kryteriów sukcesu

Celem eksploracji jest klasyfikacja stylu piwa na podstawie jego właściwości fizykochemicznych, takich jak zawartość alkoholu `(ABV)`, goryczka `(IBU)`, gęstości `(OG, FG)`, kolor `(SRM)` oraz metoda warzenia.
Docelowo rozwiązywanym problemem jest klasyfikacja wieloklasowa, ponieważ styl piwa przyjmuje wiele możliwych wartości nominalnych.
Dodatkowym celem jest zidentyfikowanie, które cechy mają największy wpływ na klasyfikację stylu piwa — będzie to realizowane m.in. przez analizę ważności cech (feature importance).

Początkowo rozważaliśmy `accuracy` (dokładność klasyfikacji) jako główną metrykę, jednak ze względu na potencjalną nierównowagę klas (niektóre style mogą występować znacznie częściej niż inne), bardziej odpowiednim wyznacznikiem jakości modelu będzie `balanced accuracy` (zrównoważona dokładność).

`Balanced accuracy` oblicza średnią arytmetyczną czułości (`recall`) dla każdej klasy, co zapewnia, że model jest oceniany sprawiedliwie, niezależnie od liczności poszczególnych stylów piwa. Dzięki temu unikniemy sytuacji, w której wysoka dokładność wynika wyłącznie z poprawnego klasyfikowania dominujących klas, podczas gdy rzadkie style są ignorowane.

Dodatkowo zastosowane zostaną następujące metryki:

**Macro F1-score** to średnia arytmetyczna wartości F1 obliczonych osobno dla każdej klasy. Traktuje wszystkie klasy z równą wagą, niezależnie od ich liczności. Jest to szczególnie przydatna metryka przy niezbalansowanych danych, ponieważ zapobiega faworyzowaniu dominujących klas.

Wzory:

- F1-score dla pojedynczej klasy:

  $$
  F1 = 2 \cdot \frac{\text{precyzja} \cdot \text{czułość}}{\text{precyzja} + \text{czułość}}
  $$

- Precyzja (precision):

  $$
  \text{precyzja} = \frac{TP}{TP + FP}
  $$
  gdzie:
  - TP (True Positive) – liczba przypadków poprawnie zaklasyfikowanych jako pozytywne (np. poprawnie rozpoznany styl piwa),
  - FP (False Positive) – liczba przypadków błędnie zaklasyfikowanych jako pozytywne (np. piwo przypisane do danego stylu, chociaż nim nie jest).

- Czułość (recall):

  $$
  \text{czułość} = \frac{TP}{TP + FN}
  $$

- Macro F1-score:

  $$
  \text{macro F1-score} = \frac{1}{N} \sum_{i=1}^{N} F1_i
  $$

  Gdzie \( N \) to liczba klas.

**Confusion matrix**

Macierz pomyłek (`confusion matrix`) pozwala przeanalizować, które style piwa są najczęściej mylone między sobą.

Sukces zostanie osiągnięty, jeżeli:
- model osiągnie `balanced accuracy` powyżej 60%,
- model osiągnie `macro F1-score` powyżej 65%.

Przy wieloklasowym problemie klasyfikacyjnym i nieidealnie zbalansowanych danych będzie to oznaczać skuteczną eksplorację stylów piwa na podstawie parametrów technicznych.

#### Dyskusja kroków dalszego postępowania



##### Dobór działania eksploracji

Zgodnie z celem eksploracji, który został zdefiniowany w Raporcie 1., dążymy do klasyfikacji stylu piwa na podstawie jego właściwości fizykochemicznych oraz identyfikacji kluczowych cech determinujących dany styl. Analiza wstępna wykazała, że atrybut `StyleID` wykazuje słabą korelację liniową z pojedynczymi cechami, co sugeruje złożony, nieliniowy charakter problemu. W związku z tym, wybrano dwuetapowe podejście algorytmiczne.

Wstępna analiza danych ujawniła trzy kluczowe wyzwania, które muszą zostać zaadresowane w dalszych krokach:

- Znaczna liczba brakujących danych w niektórych kolumnach.
- Duża liczba klas (stylów piwa), z których wiele jest niedostatecznie reprezentowanych (problem niezbalansowanych klas).
- Obecność licznych wartości odstających, które mogą zakłócać działanie algorytmów eksploracyjnych.


##### Dobór algorytmu eksploracji
1. Klasteryzacja (Grupowanie stylów piwa)

Pierwszym krokiem będzie uproszczenie problemu poprzez zastosowanie algorytmu klasteryzacji. Zamiast klasyfikować ponad 170 indywidualnych stylów, co przy niezbalansowanym zbiorze jest zadaniem niezwykle trudnym, połączymy je w mniejsze, spójne merytorycznie grupy.

- Wybrany algorytm: K-średnich (K-Means)


Algorytm K-średnich jest metodą uczenia maszynowego bez nadzoru, której celem jest podział zbioru danych na z góry określoną liczbę klastrów (k). Działa on iteracyjnie, grupując podobne do siebie punkty danych, minimalizując wariancję wewnątrz klastrów. W naszym projekcie zastosujemy go na kluczowych cechach fizykochemicznych (`ABV`, `IBU`, `OG`, `FG`, `Color`), aby zidentyfikować naturalne skupiska receptur, które dzielą podobne parametry.

Zastosowanie klasteryzacji K-średnich przed właściwą klasyfikacją przynosi kluczowe korzyści:

- Redukcja złożoności:

    Zmniejszenie liczby klas z ponad 170 do około 10-15 znacząco upraszcza zadanie klasyfikacyjne i zwiększa szansę na uzyskanie modelu o wysokiej skuteczności.

- Obsługa niezbalansowanych klas:

    Rzadkie style piwa, które mają zbyt mało próbek do efektywnego uczenia, zostaną połączone z podobnymi, liczniejszymi stylami, tworząc bardziej zrównoważone grupy.

- Zwiększenie interpretowalności:

    Grupy takie jak "Lager & Pilsner" czy "Stout & Porter" są bardziej intuicyjne i użyteczne z biznesowego punktu widzenia niż pojedyncze, często bardzo niszowe style. Wyniki klasteryzacji zostaną zweryfikowane przy użyciu wiedzy domenowej, np. w oparciu o wytyczne BJCP (Beer Judge Certification Program).

<br />

2. Klasyfikacja (Predykcja grupy stylów)

Po utworzeniu grup stylów, głównym zadaniem będzie zbudowanie modelu klasyfikacyjnego, który na podstawie cech receptury przypisze ją do odpowiedniej grupy.

- Wybrany algorytm: Las Losowy (Random Forest Classifier)


Las Losowy to zaawansowana technika uczenia zespołowego, która buduje wiele drzew decyzyjnych w procesie treningu, a ostateczną predykcję podejmuje na podstawie "głosowania" większości z nich. Jest to jeden z najskuteczniejszych i najbardziej uniwersalnych algorytmów klasyfikacyjnych.

Random Forest jest idealnym wyborem dla naszego problemu z kilku powodów:

- Wysoka skuteczność i odporność na przeuczenie: Dzięki agregacji wyników z wielu drzew, algorytm jest znacznie bardziej stabilny i dokładny niż pojedyncze drzewo decyzyjne, jednocześnie minimalizując ryzyko przeuczenia.

- Zdolność do modelowania nieliniowych zależności: Jak wskazano w Raporcie 1, proste zależności liniowe nie wystarczają do opisania stylu piwa. Lasy Losowe doskonale radzą sobie z wychwytywaniem złożonych interakcji między wieloma cechami.

- Wbudowana analiza ważności cech: Algorytm w naturalny sposób dostarcza miarę ważności każdej cechy (feature importance), co bezpośrednio realizuje nasz dodatkowy cel, jakim jest identyfikacja najważniejszych parametrów wpływających na styl piwa.

- Odporność na wartości odstające i skalowanie danych: Lasy Losowe są mniej wrażliwe na outliery, których obecność stwierdzono w Raporcie 1, i nie wymagają skomplikowanego skalowania cech.

<br />

Połączenie nienadzorowanej klasteryzacji K-średnich z nadzorowaną klasyfikacją za pomocą Lasu Losowego stanowi solidną i kompleksową strategię, która adresuje kluczowe wyzwania zidentyfikowane w naszym zbiorze danych.


##### Dobór metody testowania wyników

#### Przygotowanie danych



##### Dane brakujące i dane do ujednolicenia

Kluczowe statystyki braków (na podstawie pierwszego raportu):

|    Kolumna    | Braki | % Braków |  Decyzja |
|:-------------:|:-----:|:--------:|:--------:|
| PrimingMethod | 67101 | 90.8%    | Usunąć   |
| PrimingAmount | 69087 | 93.5%    | Usunąć   |
| PitchRate     | 39252 | 53.1%    | Usunąć   |
| MashThickness | 29864 | 40.4%    | Zachować |
| PrimaryTemp   | 22662 | 30.7%    | Zachować |
| BoilGravity   | 2990  | 4.0%     | Zachować |

Uzasadnienie decyzji:
- Kolumny z >50% braków usuwane ze względu na niemożność wiarygodnej imputacji
- `MashThickness` i `PrimaryTemp` zachowane pomimo znaczących braków - w zależności od dalszej analizy mogą być istotne dla klasyfikacji stylu piwa, a ich brak można będzie uzupełnić metodami imputacji (np. średnią, medianą lub regresją), lub później usunąć, jeśli metody imputacji nie okażą się wystarczająco wiarygodne.





##### Zamiana na nominalne/numeryczne

W naszym zbiorze danych zidentyfikowano dwie kluczowe kolumny nominalne, które wymagają konwersji: `SugarScale` oraz `BrewMethod`.

Do ich transformacji zostanie zastosowana technika kodowania etykietami (Label Encoding) poprzez zdefiniowane mapowania.

Atrybut `SugarScale`:
   - `Specific Gravity` zostanie zakodowane jako 0
   - `Plato` zostanie zakodowane jako 1

Atrybut `BrewMethod`:
   - `All Grain` zostanie zakodowane jako 0.
   - `extract` zostanie zakodowane jako 1.
   - `Partial Mash` zostanie zakodowane jako 2.
   - `BIAB` zostanie zakodowane jako 3

Dla algorytmów opartych na drzewach decyzyjnych, takich jak wybrany w naszym projekcie Las Losowy, proste kodowanie etykietami jest wystarczające i akceptowalne

##### Podzbiór danych

##### Uzupełnienie danych

#### Utworzenie modelu - Klasyfikacja

#### Utworzenie modelu - Klasteryzacja

#### Podsumowanie wyników